# Esteira Geo — Processamento Interativo

Fluxo idêntico ao executado pelo watcher: **Bronze → Silver → Gold → PostGIS**

| Área | Caminho no bronze | Quem usa |
|------|-------------------|----------|
| Exploratório | `exploratorio/<use_case>/` | Jupyter (este notebook) |
| Automatizado | `automatizado/<use_case>/` | Watcher (pipeline automático) |

Arquivos no exploratório **não são movidos** para `processados/` — podem ser reutilizados livremente.

Os arquivos-fonte do pipeline estão disponíveis em `/app/pipeline_src/` no file browser do JupyterLab.

## 0. Configuração

In [1]:
import os, sys
sys.path.insert(0, '/app/pipeline_src')  # código-fonte montado do host (editável via file browser)
sys.path.insert(0, '/app')               # módulos compilados da imagem (fallback)

# Altere aqui para processar outro use_case: enchentes_poa | enchentes_mg | enchentes_rj
# O Jupyter trabalha com bronze/exploratorio/ — o watcher usa bronze/automatizado/
os.environ['USE_CASE'] = 'enchentes_mg'

import importlib, config
importlib.reload(config)  # recarrega config com o USE_CASE atualizado

EXPLO_PREFIX = f'exploratorio/{config.USE_CASE}/'  # área de trabalho do Jupyter
AUTO_PREFIX  = f'automatizado/{config.USE_CASE}/'  # área monitorada pelo watcher

print(f"Use case    : {config.USE_CASE}")
print(f"Exploratório: s3://{config.AWS_S3_BRONZE_BUCKET}/{EXPLO_PREFIX}")
print(f"Automatizado: s3://{config.AWS_S3_BRONZE_BUCKET}/{AUTO_PREFIX}  (watcher)")
print(f"Silver      : s3://{config.AWS_S3_SILVER_BUCKET}/{config.S3_SILVER_PREFIX}")
print(f"Gold        : s3://{config.AWS_S3_GOLD_BUCKET}/{config.S3_GOLD_PREFIX}")
print(f"PostGIS     : {config.RDS_HOST}:{config.RDS_PORT}/{config.RDS_DATABASE}")
print(f"Fonte       : /app/pipeline_src/")

✓ Configuration loaded (Mode: MINIO)
  Storage: MINIO
    MinIO: http://minio:9000
    Buckets: bronze/enchentes_mg, silver/enchentes_mg, gold/enchentes_mg
  Use Case: enchentes_mg
  Database: postgis:5432/esteira_geo
  Logging: logs/pipeline.log
✓ Configuration loaded (Mode: MINIO)
  Storage: MINIO
    MinIO: http://minio:9000
    Buckets: bronze/enchentes_mg, silver/enchentes_mg, gold/enchentes_mg
  Use Case: enchentes_mg
  Database: postgis:5432/esteira_geo
  Logging: logs/pipeline.log
Use case    : enchentes_mg
Exploratório: s3://bronze/exploratorio/enchentes_mg/
Automatizado: s3://bronze/automatizado/enchentes_mg/  (watcher)
Silver      : s3://silver/enchentes_mg/
Gold        : s3://gold/enchentes_mg/
PostGIS     : postgis:5432/esteira_geo
Fonte       : /app/pipeline_src/


## 1. Inspecionar Bronze — Exploratório (S3)

In [2]:
import boto3

s3 = boto3.client(
    's3',
    endpoint_url=config.AWS_ENDPOINT_URL,
    aws_access_key_id=config.AWS_ACCESS_KEY_ID,
    aws_secret_access_key=config.AWS_SECRET_ACCESS_KEY,
    region_name=config.AWS_S3_REGION_NAME,
)

resp = s3.list_objects_v2(Bucket=config.AWS_S3_BRONZE_BUCKET, Prefix=EXPLO_PREFIX)
bronze_files = [o['Key'] for o in resp.get('Contents', []) if not o['Key'].endswith('.keep')]
print(f"{len(bronze_files)} arquivo(s) em bronze/{EXPLO_PREFIX}:")
for f in bronze_files:
    print(f"  {f}")

2 arquivo(s) em bronze/exploratorio/enchentes_mg/:
  exploratorio/enchentes_mg/citizens_minas_gerais.csv
  exploratorio/enchentes_mg/flooding_areas_minas_gerais.geojson


## 2. Silver — Normalização

> Lê de `bronze/exploratorio/<use_case>/`. Os arquivos **não são movidos** para processados — o exploratório é não-destrutivo.

In [3]:
from etl.silver_processor import process_silver

silver = process_silver(bronze_prefix=EXPLO_PREFIX, move_files=False)

if 'flooding' in silver:
    print(f"Áreas de enchente: {len(silver['flooding'])} registros")
    display(silver['flooding'].head())

if 'citizens' in silver:
    print(f"\nCidadãos: {len(silver['citizens'])} registros")
    display(silver['citizens'].head())

Áreas de enchente: 50 registros


,area_id,area_name,flood_date,severity,affected_population,regiao,geometry,normalized_date,data_quality_score
0,1,Vale do Rio Doce - Area 1,2024-03-27,medium,7606,Vale do Rio Doce,"POLYGON ((-42.00395 -19.1848, -42.01768 -19.16...",2026-03-20 20:19:59.263804,1.0
1,2,Vale do Rio Doce - Area 2,2024-09-20,very_high,7089,Vale do Rio Doce,"POLYGON ((-43.34208 -19.26508, -43.35778 -19.2...",2026-03-20 20:19:59.263804,1.0
2,3,Vale do Rio Doce - Area 3,2024-03-20,medium,5259,Vale do Rio Doce,"POLYGON ((-42.23873 -18.88689, -42.25286 -18.8...",2026-03-20 20:19:59.263804,1.0
3,4,Vale do Rio Doce - Area 4,2024-07-16,medium,2968,Vale do Rio Doce,"POLYGON ((-43.25278 -18.89919, -43.28594 -18.8...",2026-03-20 20:19:59.263804,1.0
4,5,Vale do Rio Doce - Area 5,2024-02-13,medium,956,Vale do Rio Doce,"POLYGON ((-43.18131 -19.40623, -43.20066 -19.3...",2026-03-20 20:19:59.263804,1.0



Cidadãos: 1500 registros


,citizen_id,name,address,phone,registration_date,geometry,normalized_date,data_quality_score
0,MG1132,Olivia Pires,"Av. Passos, 963",(31) 92209-8431,2023-10-17,POINT (-47.68308 -21.09232),2026-03-20 20:19:58.660847,1.0
1,MG1363,Yago Correia,"Av. Lavras, 331",(31) 98915-7129,2023-01-17,POINT (-47.12484 -21.28027),2026-03-20 20:19:58.660847,1.0
2,MG1154,Adriana Campos,"Av. Itauna, 974",(31) 92597-5321,2021-02-01,POINT (-47.0489 -20.80842),2026-03-20 20:19:58.660847,1.0
3,MG0997,Simone Machado,"Av. Centro, 582",(31) 98909-5094,2022-04-01,POINT (-47.84497 -21.40151),2026-03-20 20:19:58.660847,1.0
4,MG0550,Bruno Costa,"Rua Savassi, 907",(31) 97155-4166,2021-05-24,POINT (-44.1494 -19.59609),2026-03-20 20:19:58.660847,1.0


## 3. Gold — Batimento Geográfico (Spatial Join)

In [4]:
from etl.gold_processor import process_gold, silver_ready

has_areas, has_citizens = silver_ready()
print(f"Silver pronto — áreas: {has_areas} | cidadãos: {has_citizens}")

if has_areas and has_citizens:
    affected, unaffected, all_citizens = process_gold()

    total = len(all_citizens)
    print(f"\nResultado do batimento:")
    print(f"  Atingidos    : {len(affected)} ({len(affected)/total*100:.1f}%)")
    print(f"  Não atingidos: {len(unaffected)}")
    print(f"  Total        : {total}")

    display(affected.head())
else:
    print("Silver incompleto — execute as células anteriores primeiro.")

Silver pronto — áreas: True | cidadãos: True



Resultado do batimento:
  Atingidos    : 611 (40.7%)
  Não atingidos: 889
  Total        : 1500


,citizen_id,name,address,phone,registration_date,geometry,affected_by_flooding,affected_area_id,area_name,flood_date,severity,processing_date
4,MG0550,Bruno Costa,"Rua Savassi, 907",(31) 97155-4166,2021-05-24,POINT (-44.1494 -19.59609),True,47,Metropolitana BH - Area 8,2024-08-25,very_high,2026-03-20 20:19:59.683917
5,MG0496,Henrique Melo,"Rua Ipatinga, 508",(31) 99580-1518,2022-11-14,POINT (-45.42468 -21.68507),True,22,Sul de Minas - Area 1,2024-02-24,medium,2026-03-20 20:19:59.683917
6,MG0375,Eduardo Ferreira,"Rua Pampulha, 863",(31) 94284-6599,2021-07-12,POINT (-42.86066 -19.79398),True,8,Vale do Rio Doce - Area 9,2024-07-25,very_high,2026-03-20 20:19:59.683917
7,MG0014,Juliana Alves,"Rua Itajuba, 107",(31) 94763-1830,2024-03-08,POINT (-45.30298 -22.18546),True,24,Sul de Minas - Area 3,2024-03-20,medium,2026-03-20 20:19:59.683917
8,MG0577,Camila Azevedo,"Rua Formiga, 109",(31) 95521-4974,2021-02-25,POINT (-48.22636 -19.65192),True,34,Triangulo Mineiro - Area 3,2024-01-18,medium,2026-03-20 20:19:59.683917


## 4. PostGIS — Sincronização

In [5]:
from etl.postgis_loader import load_to_postgis

load_to_postgis(sync_areas=has_areas, sync_citizens=(has_areas and has_citizens))
print("PostGIS sincronizado.")

PostGIS sincronizado.


## 5. Consultas no PostGIS

In [6]:
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine(
    f'postgresql+psycopg2://{config.RDS_USER}:{config.RDS_PASSWORD}'
    f'@{config.RDS_HOST}:{config.RDS_PORT}/{config.RDS_DATABASE}'
)

use_case = config.USE_CASE

with engine.connect() as conn:
    stats = pd.read_sql(f"""
        SELECT
            COUNT(*) AS total,
            SUM(CASE WHEN affected_by_flooding THEN 1 ELSE 0 END) AS atingidos,
            SUM(CASE WHEN NOT affected_by_flooding THEN 1 ELSE 0 END) AS nao_atingidos
        FROM {use_case}_citizens
    """, conn)
    display(stats)

    sample = pd.read_sql(f"""
        SELECT citizen_id, name, affected_by_flooding, ST_AsText(geometry) AS geom
        FROM {use_case}_citizens
        WHERE affected_by_flooding = TRUE
        LIMIT 5
    """, conn)
    display(sample)


,total,atingidos,nao_atingidos
0,1500,611,889


,citizen_id,name,affected_by_flooding,geom
0,MG0550,Bruno Costa,True,POINT(-44.149395 -19.596093)
1,MG0496,Henrique Melo,True,POINT(-45.424677 -21.685069)
2,MG0375,Eduardo Ferreira,True,POINT(-42.860659 -19.793977)
3,MG0014,Juliana Alves,True,POINT(-45.302977 -22.185459)
4,MG0577,Camila Azevedo,True,POINT(-48.226359 -19.651916)


## 6. Visualização no Mapa (Leaflet via IFrame)

In [7]:
from IPython.display import IFrame

# Abre o mapa do Flask (certifique-se que o serviço web está rodando na porta 5000)
IFrame(src=f'http://localhost:5000/map?use_case={config.USE_CASE}', width='100%', height=600)

## 7. Upload para o Exploratório

Faz upload de um arquivo local para `bronze/exploratorio/<use_case>/` e reprocessa interativamente.
O arquivo **não é movido** para `processados/` — pode ser reutilizado quantas vezes quiser.

In [8]:
from etl.silver_processor import process_silver
from etl.gold_processor import process_gold, silver_ready
from etl.postgis_loader import load_to_postgis

# Ajuste o caminho para o arquivo que deseja explorar
local_file = '/data/bronze/automatizado/enchentes_poa/citizens_sample.csv'

# Destino: exploratorio/<use_case>/<filename>
s3_key = f"exploratorio/{config.USE_CASE}/{os.path.basename(local_file)}"
s3.upload_file(local_file, config.AWS_S3_BRONZE_BUCKET, s3_key)
print(f"Upload: s3://{config.AWS_S3_BRONZE_BUCKET}/{s3_key}")

# Reprocessar a partir do exploratório (não move arquivos)
silver = process_silver(bronze_prefix=EXPLO_PREFIX, move_files=False)
has_areas, has_citizens = silver_ready()
if has_areas and has_citizens:
    affected, unaffected, all_citizens = process_gold()
    load_to_postgis(sync_areas=True, sync_citizens=True)
    print(f"Reprocessado: {len(affected)} atingidos / {len(all_citizens)} total")

Upload: s3://bronze/exploratorio/enchentes_mg/citizens_sample.csv


Reprocessado: 611 atingidos / 1550 total
